In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])

In [2]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Python executable:", sys.executable)
print("Transformers:", transformers.__version__, transformers.__file__)
print("Accelerate:", accelerate.__version__, accelerate.__file__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)

2.9.0+cu128
True
Tesla T4


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from google.colab import drive


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

TRAIN_SENTENCE_LOG = "/content/drive/MyDrive/RoBERTa_SingleStep_train_sentence_log.csv"
VAL_SENTENCE_LOG = "/content/drive/MyDrive/RoBERTa_SingleStep_val_sentence_log.csv"
TEST_SENTENCE_LOG = "/content/drive/MyDrive/RoBERTa_SingleStep_test_sentence_log.csv"


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SEED = 42

EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]


# ------------------------------------------------------------
# Load BRIGHTER dataset
# ------------------------------------------------------------

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()


# ------------------------------------------------------------
# Remove disgust and create single-step labels
# ------------------------------------------------------------

def convert_single_step_labels(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        for level in LEVELS:
            df[f"{emotion}_{level}"] = (
                df[emotion] == level
            ).astype(int)

    return df


# ------------------------------------------------------------
# Plot function
# ------------------------------------------------------------

def plot_class_distribution(train_set, val_set, test_set, title):
    train_counts = train_set[LABELS].sum()
    val_counts = val_set[LABELS].sum()
    test_counts = test_set[LABELS].sum()

    x = np.arange(len(LABELS))
    width = 0.27

    plt.figure(figsize=(18, 6))

    plt.bar(
        x - width,
        train_counts,
        width,
        label="Train"
    )

    plt.bar(
        x,
        val_counts,
        width,
        label="Validation"
    )

    plt.bar(
        x + width,
        test_counts,
        width,
        label="Test"
    )

    plt.xticks(
        x,
        LABELS,
        rotation=45,
        ha="right"
    )

    plt.xlabel("Emotion–Intensity Labels")
    plt.ylabel("Number of Samples")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# Plot original BRIGHTER distribution before 70/20/10 split
# ------------------------------------------------------------

original_train_single = convert_single_step_labels(train_df)
original_val_single = convert_single_step_labels(val_df)
original_test_single = convert_single_step_labels(test_df)

plot_class_distribution(
    original_train_single,
    original_val_single,
    original_test_single,
    "Original BRIGHTER Class Distribution Before 70/20/10 Split"
)


# ------------------------------------------------------------
# Combine and create new 70/20/10 split
# ------------------------------------------------------------

full_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

total_size = len(full_df)

train_size = int(0.70 * total_size)
val_size = int(0.20 * total_size)

train_df = full_df.iloc[
    :train_size
].reset_index(drop=True)

val_df = full_df.iloc[
    train_size:train_size + val_size
].reset_index(drop=True)

test_df = full_df.iloc[
    train_size + val_size:
].reset_index(drop=True)

print("New split sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


# ------------------------------------------------------------
# Create labels for new 70/20/10 splits
# ------------------------------------------------------------

train_single = convert_single_step_labels(train_df)
val_single = convert_single_step_labels(val_df)
test_single = convert_single_step_labels(test_df)


# ------------------------------------------------------------
# Distribution after 70/20/10 split
# ------------------------------------------------------------

train_counts = train_single[LABELS].sum()
val_counts = val_single[LABELS].sum()
test_counts = test_single[LABELS].sum()

distribution_df = pd.DataFrame({
    "Train": train_counts,
    "Validation": val_counts,
    "Test": test_counts
})

distribution_df["Total"] = (
    distribution_df["Train"]
    + distribution_df["Validation"]
    + distribution_df["Test"]
)

print("\nClass distribution after 70/20/10 split:")
print(distribution_df)

plot_class_distribution(
    train_single,
    val_single,
    test_single,
    "Class Distribution for Single-Step Classification – 70/20/10 Split"
)


# ------------------------------------------------------------
# Create sentence-wise true-label logs
# ------------------------------------------------------------

def create_sentence_log(df):
    rows = []

    for sentence_id, row in df.iterrows():

        true_labels = [
            label
            for label in LABELS
            if row[label] == 1
        ]

        rows.append({
            "sentence_id": sentence_id,
            "sentence": row["text"],
            "true_labels": (
                ", ".join(true_labels)
                if true_labels
                else "No Emotion"
            )
        })

    return pd.DataFrame(rows)


train_sentence_log = create_sentence_log(train_single)
val_sentence_log = create_sentence_log(val_single)
test_sentence_log = create_sentence_log(test_single)


# ------------------------------------------------------------
# Save sentence logs to Google Drive
# ------------------------------------------------------------

train_sentence_log.to_csv(
    TRAIN_SENTENCE_LOG,
    index=False
)

val_sentence_log.to_csv(
    VAL_SENTENCE_LOG,
    index=False
)

test_sentence_log.to_csv(
    TEST_SENTENCE_LOG,
    index=False
)

print("\nSentence logs saved successfully:")
print(TRAIN_SENTENCE_LOG)
print(VAL_SENTENCE_LOG)
print(TEST_SENTENCE_LOG)

print("\nTrain sentence log sample:")
print(train_sentence_log.head())